# Basic RAG: Company Documentation Q&A

**Objective:** Load documents, create chunks and embeddings, retrieve relevant passages with semantic search, and evaluate chunk sizes of 200, 500, and 1,000 characters.

The included handbook is synthetic sample data. Replace it only with company documents you are authorized to use.

## 1. Install packages

In [ ]:
%pip install -q sentence-transformers chromadb langchain-text-splitters pandas

## 2. Load the document

In [ ]:
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

doc_path = Path("data/company_handbook.txt")
document_text = doc_path.read_text(encoding="utf-8")
print("Characters loaded:", len(document_text))
print(document_text[:500])

## 3. Load an embedding model
Embeddings represent text as vectors. Semantic search compares the meaning of the question with document chunks.

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model ready")

## 4. Define questions and chunking/retrieval functions

In [ ]:
questions = [
    "How many days in advance should planned leave be requested?",
    "How do employees report laptop or network problems?",
    "What is the deadline for submitting expense claims?",
    "What should employees do with suspicious emails?",
    "What are the normal working hours?"
]

def split_text(text, chunk_size, overlap=40):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=min(overlap, max(0, chunk_size // 5)),
        separators=["\\n\\n", "\\n", ". ", " ", ""]
    )
    return splitter.split_text(text)

def retrieve(chunks, question, top_k=3):
    vectors = embedding_model.encode(chunks, normalize_embeddings=True)
    q_vector = embedding_model.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ q_vector
    order = scores.argsort()[::-1][:min(top_k, len(chunks))]
    return [(chunks[i], float(scores[i])) for i in order]

def answer_from_context(retrieved):
    # Transparent extractive baseline; this does not pretend to be an LLM-generated answer.
    if not retrieved:
        return "No relevant information found."
    return "\\n---\\n".join(text for text, score in retrieved)

## 5. Compare chunk sizes
The similarity score is a retrieval signal, not a guarantee that the answer is correct. Review the retrieved text and fill the human evaluation columns after running.

In [ ]:
rows = []
for size in [200, 500, 1000]:
    chunks = split_text(document_text, size)
    for question in questions:
        hits = retrieve(chunks, question, top_k=3)
        rows.append({
            "chunk_size_chars": size,
            "question": question,
            "number_of_chunks": len(chunks),
            "top1_cosine_similarity": round(hits[0][1], 4) if hits else None,
            "retrieved_context": "\\n---\\n".join(t for t, score in hits),
            "baseline_answer": answer_from_context(hits),
            "relevance_1_to_5": None,
            "completeness_1_to_5": None,
            "groundedness_1_to_5": None
        })
results = pd.DataFrame(rows)
display(results)
results.to_csv("rag_chunk_comparison_results.csv", index=False)

## 6. Optional: use ChromaDB as the vector database

In [ ]:
def chroma_search(chunk_size, question, top_k=3):
    chunks = split_text(document_text, chunk_size)
    client = chromadb.Client()
    name = f"handbook_chunks_{chunk_size}"
    try:
        client.delete_collection(name)
    except Exception:
        pass
    collection = client.create_collection(name, metadata={"hnsw:space": "cosine"})
    vectors = embedding_model.encode(chunks, normalize_embeddings=True).tolist()
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=vectors
    )
    qv = embedding_model.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=qv, n_results=min(top_k, len(chunks)))
    return list(zip(result["documents"][0], result["distances"][0]))

chroma_search(500, questions[0])

## 7. Optional LLM generation
The core experiment above compares retrieval using an extractive baseline. To generate fluent answers with an LLM, configure an approved provider. Never commit API keys to GitHub. Example with OpenAI-compatible LangChain client:

In [ ]:
# Optional: uncomment after installing langchain-openai and setting OPENAI_API_KEY
# %pip install -q langchain-openai
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
#
# def generate_answer(question, retrieved):
#     context = "\\n\\n".join(text for text, score in retrieved)
#     prompt = (
#         "Answer using only the context. If the answer is missing, say you don't know.\\n\\n"
#         f"Context:\\n{context}\\n\\nQuestion: {question}"
#     )
#     return llm.invoke(prompt).content
#
# hits = retrieve(split_text(document_text, 500), questions[0])
# print(generate_answer(questions[0], hits))

## 8. Evaluation and conclusion
Rate each output from 1 (poor) to 5 (excellent): **relevance**, **completeness**, and **groundedness**. Groundedness means the answer is supported by the retrieved source. Do not choose a winning chunk size until you have reviewed actual results.

**Conclusion template:** “Chunk size ___ performed best for this document and question set because ___. The evidence was ___ (actual ratings and observations). Results may differ on other documents.”

## 9. Troubleshooting and submission checklist
- Run cells in order; the embedding model downloads on first use and requires internet access.
- Use approved documents only; do not upload confidential information or secrets.
- Add screenshots of document loading, retrieved passages, comparison output, and completed evaluation table.
- Push the notebook, README, report, and permitted sample data to GitHub.